# Comprehensive Cryptocurrency Market Analysis
## Complete Analysis Pipeline

This notebook provides a complete analysis of cryptocurrency market data and the Fear & Greed Index, covering everything from data loading to model development.

## 1. Environment Setup and Data Loading

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime, timedelta

# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb

# Time Series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Set plot style
plt.style.use('seaborn')
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

## 2. Data Loading and Initial Exploration

In [ ]:
# Load the datasets
data_dir = Path('../data/raw/')
historical_data = pd.read_csv(data_dir / 'historical_data.csv')
fear_greed_data = pd.read_csv(data_dir / 'fear_greed_index.csv')

# Display basic information
print("Historical Data Shape:", historical_data.shape)
print("\nFear & Greed Data Shape:", fear_greed_data.shape)

print("\nHistorical Data Info:")
print(historical_data.info())

print("\nFear & Greed Data Info:")
print(fear_greed_data.info())

# Display first few rows
print("\nHistorical Data Head:")
display(historical_data.head())

print("\nFear & Greed Data Head:")
display(fear_greed_data.head())

## 3. Data Cleaning and Preprocessing

In [ ]:
def clean_historical_data(df):
    """Clean and preprocess historical price data"""
    df = df.copy()
    
    # Convert date column to datetime
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date')
    
    # Handle missing values
    df = df.ffill()  # Forward fill for OHLC data
    
    # Calculate returns and other metrics
    if 'close' in df.columns:
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
    
    return df

def clean_fear_greed_data(df):
    """Clean and preprocess Fear & Greed Index data"""
    df = df.copy()
    
    # Convert date column to datetime
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date')
    
    # Forward fill missing values
    df = df.ffill()
    
    return df

# Apply cleaning functions
historical_clean = clean_historical_data(historical_data)
fear_greed_clean = clean_fear_greed_data(fear_greed_data)

# Merge datasets
if 'date' in historical_clean.columns and 'date' in fear_greed_clean.columns:
    merged_data = pd.merge(historical_clean, fear_greed_clean, on='date', how='left')
    print("\nMerged Data Shape:", merged_data.shape)
    display(merged_data.head())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Time series plot of closing prices
plt.figure(figsize=(14, 6))
plt.plot(merged_data['date'], merged_data['close'])
plt.title('Cryptocurrency Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(True)
plt.show()

# Fear & Greed Index over time
if 'fear_greed' in merged_data.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(merged_data['date'], merged_data['fear_greed'])
    plt.title('Fear & Greed Index Over Time')
    plt.xlabel('Date')
    plt.ylabel('Index Value')
    plt.grid(True)
    plt.show()

# Correlation heatmap
plt.figure(figsize=(12, 8))
correlation = merged_data.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 5. Feature Engineering

In [ ]:
def create_features(df):
    """Create technical indicators and other features"""
    df = df.copy()
    
    # Moving Averages
    df['sma_7'] = df['close'].rolling(window=7).mean()
    df['sma_21'] = df['close'].rolling(window=21).mean()
    
    # Volatility
    df['volatility'] = df['returns'].rolling(window=21).std() * np.sqrt(252)
    
    # Price range
    if all(col in df.columns for col in ['high', 'low']):
        df['daily_range'] = (df['high'] - df['low']) / df['close']
    
    # Lagged features
    for lag in [1, 2, 3, 5, 7]:
        df[f'return_lag_{lag}'] = df['returns'].shift(lag)
    
    # Moving average convergence divergence (MACD)
    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = df['ema_12'] - df['ema_26']
    
    # Relative Strength Index (RSI)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    df['bb_upper'] = df['sma_21'] + (2 * df['close'].rolling(window=21).std())
    df['bb_lower'] = df['sma_21'] - (2 * df['close'].rolling(window=21).std())
    
    # Drop NaN values created by rolling windows
    df = df.dropna()
    
    return df

# Apply feature engineering
featured_data = create_features(merged_data)
print("Data with features shape:", featured_data.shape)
featured_data.head()

## 6. Time Series Analysis

In [ ]:
# Time series decomposition
if len(featured_data) > 0:
    # Resample to daily data if needed
    daily_data = featured_data.set_index('date').resample('D').last()
    
    # Decompose the time series
    decomposition = seasonal_decompose(daily_data['close'].ffill(), period=30)  # 30-day period
    
    # Plot decomposition
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 12))
    
    # Original series
    ax1.plot(decomposition.observed)
    ax1.set_ylabel('Observed')
    
    # Trend
    ax2.plot(decomposition.trend)
    ax2.set_ylabel('Trend')
    
    # Seasonality
    ax3.plot(decomposition.seasonal)
    ax3.set_ylabel('Seasonal')
    
    # Residuals
    ax4.plot(decomposition.resid)
    ax4.set_ylabel('Residuals')
    
    plt.tight_layout()
    plt.show()
    
    # Check stationarity with Augmented Dickey-Fuller test
    print("\nAugmented Dickey-Fuller Test for Stationarity:")
    result = adfuller(daily_data['close'].dropna())
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value}')

## 7. Model Development

In [ ]:
def prepare_model_data(df, target_col='close', test_size=0.2):
    """Prepare data for modeling"""
    # Create target variable (next day's close price)
    df = df.copy()
    df['target'] = df[target_col].shift(-1)
    
    # Drop rows with missing values
    df = df.dropna()
    
    # Select features (exclude date and target)
    feature_cols = [col for col in df.columns if col not in ['date', 'target']]
    X = df[feature_cols]
    y = df['target']
    
    # Split into train and test sets (time-based split)
    split_idx = int(len(df) * (1 - test_size))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols

# Prepare data for modeling
X_train, X_test, y_train, y_test, feature_cols = prepare_model_data(featured_data)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

# Train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feature_importance.head(15))
plt.title('Top 15 Important Features')
plt.show()

## 8. Model Evaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Evaluate model performance"""
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    metrics = {
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train MAE': mean_absolute_error(y_train, y_train_pred),
        'Test MAE': mean_absolute_error(y_test, y_test_pred),
        'Train R2': r2_score(y_train, y_train_pred),
        'Test R2': r2_score(y_test, y_test_pred)
    }
    
    # Plot actual vs predicted
    plt.figure(figsize=(14, 6))
    
    # Plot training data
    plt.subplot(1, 2, 1)
    plt.scatter(y_train, y_train_pred, alpha=0.5)
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Training Data: Actual vs Predicted')
    
    # Plot test data
    plt.subplot(1, 2, 2)
    plt.scatter(y_test, y_test_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Test Data: Actual vs Predicted')
    
    plt.tight_layout()
    plt.show()
    
    return metrics

# Evaluate the model
metrics = evaluate_model(rf_model, X_train, X_test, y_train, y_test)
pd.DataFrame([metrics])

## 9. Model Comparison

In [ ]:
def train_compare_models(X_train, X_test, y_train, y_test):
    """Train and compare multiple models"""
    models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    }
    
    results = []
    
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        metrics = {
            'Model': name,
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        }
        
        results.append(metrics)
    
    return pd.DataFrame(results)

# Compare models
model_comparison = train_compare_models(X_train, X_test, y_train, y_test)
model_comparison

## 10. Trading Strategy Backtesting

In [ ]:
def backtest_strategy(data, model, feature_cols, initial_capital=10000, commission=0.001):
    """Backtest a simple trading strategy using model predictions"""
    # Make sure we have the necessary columns
    if not all(col in data.columns for col in feature_cols + ['date', 'close']):
        print("Missing required columns in the data")
        return None
    
    # Prepare features
    X = data[feature_cols]
    
    # Make predictions
    data['predicted_return'] = model.predict(X)
    
    # Create signals
    data['signal'] = 0  # 0 = hold, 1 = buy, -1 = sell
    data.loc[data['predicted_return'] > 0.01, 'signal'] = 1  # Buy if predicted return > 1%
    data.loc[data['predicted_return'] < -0.01, 'signal'] = -1  # Sell if predicted return < -1%
    
    # Calculate daily returns
    data['daily_return'] = data['close'].pct_change()
    data['strategy_return'] = data['signal'].shift(1) * data['daily_return']
    
    # Account for commission
    data['position_change'] = data['signal'].diff().fillna(0).abs()
    data['strategy_return'] = data['strategy_return'] - (data['position_change'] * commission)
    
    # Calculate cumulative returns
    data['cumulative_market'] = (1 + data['daily_return']).cumprod()
    data['cumulative_strategy'] = (1 + data['strategy_return']).cumprod()
    
    # Plot results
    plt.figure(figsize=(14, 7))
    plt.plot(data['date'], data['cumulative_market'], label='Buy & Hold')
    plt.plot(data['date'], data['cumulative_strategy'], label='Trading Strategy')
    plt.title('Strategy Backtesting Results')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Calculate performance metrics
    total_return = data['cumulative_strategy'].iloc[-1] - 1
    annualized_return = (1 + total_return) ** (252/len(data)) - 1
    annualized_volatility = data['strategy_return'].std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_volatility
    
    metrics = {
        'Total Return': total_return,
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_volatility,
        'Sharpe Ratio': sharpe_ratio
    }
    
    return pd.DataFrame([metrics])

# Run backtest
if len(feature_cols) > 0:
    backtest_results = backtest_strategy(
        featured_data.copy(), 
        rf_model, 
        feature_cols
    )
    
    if backtest_results is not None:
        print("\nBacktesting Results:")
        display(backtest_results)

## 11. Conclusion and Next Steps

### Key Findings:
1. The model shows promising results in predicting cryptocurrency price movements.
2. The most important features for prediction are [list top features].
3. The trading strategy [outperforms/underperforms] the buy-and-hold strategy.

### Next Steps:
1. **Hyperparameter Tuning**: Optimize model parameters using grid search or Bayesian optimization.
2. **Feature Engineering**: Try additional technical indicators or external data sources.
3. **Advanced Models**: Experiment with deep learning models like LSTMs or Transformers.
4. **Risk Management**: Implement stop-loss and take-profit mechanisms.
5. **Live Testing**: Test the strategy with paper trading before real money.

### Limitations:
1. Past performance is not indicative of future results.
2. The model may not account for black swan events.
3. Transaction costs and slippage can impact real-world performance.